In [67]:
import pandas as pd
import numpy as np
import pickle 
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [68]:
data = pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [69]:
data = data.drop(["RowNumber","CustomerId","Surname"],axis=1)
data


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [70]:
le = LabelEncoder()
data['Gender'] = le.fit_transform(data['Gender'])


In [71]:
ohe = OneHotEncoder(handle_unknown='ignore')
geo_encoded = ohe.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=ohe.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [72]:
data = pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [73]:
# splitting the data into independent and target features
X = data.drop('EstimatedSalary',axis=1)
y = data['EstimatedSalary']

In [74]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [75]:
#saving the encoder and scalar for later use

with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(le,file)

with open('onehot_encoding_geo.pkl','wb') as file:
    pickle.dump(ohe,file)

with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

#### regression problem statement


In [76]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense



In [77]:
#build the model
model = Sequential([
    Dense(64,activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32,activation='relu'),
    Dense(1)
])

In [78]:
from streamlit import metric
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])

model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_9 (Dense)             (None, 64)                832       
                                                                 
 dense_10 (Dense)            (None, 32)                2080      
                                                                 
 dense_11 (Dense)            (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [79]:
from datetime import datetime
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
##setup the tensorboard

log_dir = "Regressionlogs/fit/" + datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

In [80]:
##early stopping

early_stopping_callback = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)


In [81]:
#train the model

history = model.fit(
    X_train,y_train,
    validation_data=[X_test,y_test],
    epochs = 100,
    callbacks = [early_stopping_callback,tensorflow_callback])

Epoch 1/100
250/250 [==============================] - 1s 3ms/step - loss: 100377.1172 - mae: 100377.1172 - val_loss: 98514.5938 - val_mae: 98514.5938
Epoch 2/100
250/250 [==============================] - 1s 2ms/step - loss: 99628.7031 - mae: 99628.7031 - val_loss: 97007.0703 - val_mae: 97007.0703
Epoch 3/100
250/250 [==============================] - 0s 2ms/step - loss: 97032.2109 - mae: 97032.2109 - val_loss: 93199.6797 - val_mae: 93199.6797
Epoch 4/100
250/250 [==============================] - 0s 2ms/step - loss: 91933.6094 - mae: 91933.6094 - val_loss: 86840.7422 - val_mae: 86840.7422
Epoch 5/100
250/250 [==============================] - 1s 2ms/step - loss: 84512.7422 - mae: 84512.7422 - val_loss: 78699.0312 - val_mae: 78699.0312
Epoch 6/100
250/250 [==============================] - 1s 2ms/step - loss: 75706.7578 - mae: 75706.7578 - val_loss: 70176.2734 - val_mae: 70176.2734
Epoch 7/100
250/250 [==============================] - 1s 2ms/step - loss: 67109.9922 - mae: 67109.9922 

In [82]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [83]:
%reload_ext tensorboard

%tensorboard --logdir Regressionlogs\fit

Reusing TensorBoard on port 6006 (pid 18372), started 4:01:58 ago. (Use '!kill 18372' to kill it.)

In [84]:
test_loss,test_mae = model.evaluate(X_test,y_test)
print(test_loss,test_mae)

63/63 [==============================] - 0s 2ms/step - loss: 50281.0742 - mae: 50281.0742
50281.07421875 50281.07421875


In [85]:
model.save('regression_model.h5')

d:\mohit\AIML\ann classifier\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
